# Colab runner – Phân loại rác 3 nhóm (SPEC v2)

Notebook này **chỉ gọi CLI** trong repo (`python -m src.engine.<module>`); mọi logic nằm trong `src/`.
Không commit output của notebook.

| Mục | Người | Việc |
|---|---|---|
| 0 | tất cả | Cấu hình, lấy code, giải nén dữ liệu, chạy test + smoke |
| 1 | SV2 | Leakage ablation (sạch vs rò rỉ) và flat vs fine_grained trên `convnext_tiny` |
| 2 | SV3 | So 3 backbone + augmentation cơ bản vs nâng cao, chốt `best_model.pth` |
| 3 | SV4 | Hiệu chuẩn T, τ → đánh giá 4 tầng → Grad-CAM theo nhóm lỗi |

Chuẩn bị: trên Drive có `MyDrive/PhanLoaiRac/waste_data_v2.zip`. `Runtime → Change runtime type → T4 GPU`.
Mỗi lần train đã có `summary.json` trên Drive sẽ được **bỏ qua** → Colab ngắt thì chỉ cần chạy lại.

In [ ]:
# ==== 0. CẤU HÌNH ====
REPO_URL   = 'https://github.com/<ten-nhom>/vn-waste-classification-cv.git'   # sửa thành repo của nhóm
BRANCH     = 'main'
CODE_ZIP   = ''          # nếu chưa có GitHub: tải zip mã nguồn lên Drive và ghi đường dẫn vào đây
DRIVE_DIR  = '/content/drive/MyDrive/PhanLoaiRac'
ZIP_NAME   = 'waste_data_v2.zip'
RUNS       = f'{DRIVE_DIR}/runs'                  # checkpoint + kết quả lưu trên Drive
REPO_DIR   = '/content/vn-waste-classification-cv'
DATA_ROOT  = '/content/data'                      # ảnh giải nén ra SSD của Colab
MANIFEST   = '/content/manifest_colab.csv'
VN_REAL_MANIFEST = f'{DRIVE_DIR}/vn_real/vn_real_manifest.csv'   # tầng 4 (bỏ qua nếu chưa có)
VN_REAL_ROOT     = f'{DRIVE_DIR}/vn_real'
EPOCHS     = 18
BATCH      = 64

In [ ]:
import os, json, glob, shutil, zipfile, time
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(RUNS, exist_ok=True)
if CODE_ZIP:
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    with zipfile.ZipFile(CODE_ZIP) as z: z.extractall('/content')
elif os.path.isdir(REPO_DIR):
    get_ipython().system(f'git -C {REPO_DIR} pull -q')
else:
    get_ipython().system(f'git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}')
os.chdir(REPO_DIR)
get_ipython().system('pip install -q -r requirements-colab.txt')
# ---- Kiểm tra GPU: dừng ngay nếu phiên Colab không có GPU (train bằng CPU chậm hơn ~30 lần) ----
import torch
get_ipython().system('nvidia-smi -L')
if not torch.cuda.is_available():
    raise SystemExit('KHÔNG có GPU! Runtime → Change runtime type → chọn T4 GPU → Save, '
                     'rồi chạy lại notebook từ ô đầu. Nếu Colab báo hết hạn mức GPU thì phải chờ hoặc dùng tài khoản khác.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Giải nén dữ liệu ra ổ cục bộ (chỉ lần đầu mỗi phiên Colab)
if not os.path.exists(MANIFEST):
    t = time.time()
    shutil.copy(f'{DRIVE_DIR}/{ZIP_NAME}', '/content/data.zip')    # copy 1 file lớn nhanh hơn đọc lẻ từ Drive
    with zipfile.ZipFile('/content/data.zip') as z:
        z.extractall('/content')                                  # -> /content/data/... + /content/manifest_colab.csv
    os.remove('/content/data.zip')
    print(f'Giải nén xong sau {time.time() - t:.0f}s')
COMMON = f'--manifest {MANIFEST} --data_root {DATA_ROOT}'
VN_ARGS = f'--vn_real_manifest {VN_REAL_MANIFEST} --vn_real_root {VN_REAL_ROOT}' if os.path.exists(VN_REAL_MANIFEST) else ''
print('VN-Real:', 'có' if VN_ARGS else 'chưa có (bỏ qua tầng 4)')

def sh(cmd):
    print('$', cmd); get_ipython().system(cmd)

def train(backbone='convnext_tiny', mode='flat', leaky=False, aug='advanced', loss='weighted_ce', extra=''):
    name = f"{backbone}_{mode}_{'leaky' if leaky else 'clean'}_{aug}_{loss}"
    if os.path.exists(f'{RUNS}/{name}/summary.json'):
        print('Đã có, bỏ qua:', name); return name
    sh(f'python -m src.engine.train {COMMON} --out_dir {RUNS} --backbone {backbone} --mode {mode} '
       f'--aug {aug} --loss {loss} --epochs {EPOCHS} --batch {BATCH} {"--allow_leaky" if leaky else ""} {extra}')
    return name

def summary(name):
    return json.load(open(f'{RUNS}/{name}/summary.json'))

In [ ]:
# Test đơn vị + chạy thử toàn pipeline với vài ảnh (bắt lỗi code, KHÔNG đo chất lượng) - vài phút
sh(f'WASTE_MANIFEST={MANIFEST} WASTE_DATA_ROOT={DATA_ROOT} python -m pytest tests -q')
sh(f'python -m src.engine.train {COMMON} --out_dir /content/smoke --backbone convnext_tiny --mode fine_grained --smoke')
ck = glob.glob('/content/smoke/*/best_model.pth')[0]
sh(f'python -m src.engine.calibrate --ckpt {ck} {COMMON} --smoke')
sh(f'python -m src.engine.evaluate --ckpt {ck} {COMMON} {VN_ARGS} --smoke --bootstrap 50')

## 1. SV2 – Leakage ablation và flat vs fine_grained (`convnext_tiny`)
- **clean**: train/val/test theo cột `split` (đã khử trùng). **leaky**: theo `split_raw` (chia ngẫu nhiên, giữ ảnh trùng).
- So `test_in_own_split` (leaky bị "ảo") với `test_shift` (trung thực cho cả hai).
- Ngay sau run đầu tiên: đưa `runs/convnext_tiny_flat_clean_advanced_weighted_ce/best_model.pth` cho SV4/SV5 làm **checkpoint tạm**.

In [ ]:
r_clean = train('convnext_tiny', 'flat')
r_leaky = train('convnext_tiny', 'flat', leaky=True)
r_fine  = train('convnext_tiny', 'fine_grained')
import pandas as pd
rows = []
for n in (r_clean, r_leaky, r_fine):
    s = summary(n)
    rows.append(dict(run=n, val_bal_acc=s['best_val_bal_acc'], test_in_bal_acc=s['test_in_own_split']['bal_acc'],
                     test_in_macro_f1=s['test_in_own_split']['macro_f1'], test_shift_bal_acc=s['test_shift']['bal_acc'],
                     n_train=s['n_train'], phut=round(s['train_minutes'], 1)))
t = pd.DataFrame(rows).set_index('run'); t['gap_in_shift'] = t.test_in_bal_acc - t.test_shift_bal_acc
display(t.round(4))
BEST_MODE = 'fine_grained' if summary(r_fine)['best_val_bal_acc'] >= summary(r_clean)['best_val_bal_acc'] else 'flat'
print('Chế độ tốt hơn trên val:', BEST_MODE)

## 2. SV3 – So 3 backbone và augmentation (trên chế độ tốt nhất của SV2)
Chọn model cuối theo **val bal_acc** (không nhìn test). Bảng có số tham số và độ trễ CPU/GPU.

In [ ]:
for bb in ['mobilenetv3_large_100', 'convnext_tiny', 'swinv2_tiny_window8_256']:
    train(bb, BEST_MODE)
train('convnext_tiny', BEST_MODE, aug='basic')
sh(f'python -m src.engine.benchmark --runs_dir {RUNS} --latency')
tab = pd.read_csv(f'{RUNS}/runs_table.csv')
display(tab)
cand = tab[(~tab.leaky)].sort_values('val_bal_acc', ascending=False)
BEST_RUN = cand.iloc[0]['run']
print('Model chốt (val bal_acc cao nhất):', BEST_RUN)
os.makedirs(f'{DRIVE_DIR}/final', exist_ok=True)
shutil.copy(f'{RUNS}/{BEST_RUN}/best_model.pth', f'{DRIVE_DIR}/final/best_model.pth')

## 3. SV4 – Hiệu chuẩn, đánh giá 4 tầng, Grad-CAM
`calibrate` → `temperature.json`; `evaluate` → `results.json`, `summary.csv`, ma trận nhầm lẫn, risk–coverage từng tầng;
thêm 1 lần eval bằng center-crop để so với letterbox; `gradcam` → ảnh theo 4 nhóm lỗi.

In [ ]:
CK = f'{RUNS}/{BEST_RUN}/best_model.pth'
sh(f'python -m src.engine.calibrate --ckpt {CK} {COMMON}')
sh(f'python -m src.engine.evaluate --ckpt {CK} {COMMON} {VN_ARGS}')
sh(f'python -m src.engine.evaluate --ckpt {CK} {COMMON} {VN_ARGS} --eval_mode center_crop --bootstrap 200')
vn_root = f'--vn_real_root {VN_REAL_ROOT}' if VN_ARGS else ''
sh(f'python -m src.explain.gradcam --ckpt {CK} --predictions {RUNS}/{BEST_RUN}/eval_letterbox/predictions.csv '
   f'--data_root {DATA_ROOT} {vn_root}')
shutil.copy(f'{RUNS}/{BEST_RUN}/temperature.json', f'{DRIVE_DIR}/final/temperature.json')

In [ ]:
from IPython.display import Image as Img
ev = f'{RUNS}/{BEST_RUN}/eval_letterbox'
display(pd.read_csv(f'{ev}/summary.csv', index_col=0).round(4))
lc = pd.read_csv(f'{RUNS}/{BEST_RUN}/eval_center_crop/summary.csv', index_col=0)
display(pd.DataFrame({'letterbox': pd.read_csv(f'{ev}/summary.csv', index_col=0).bal_acc, 'center_crop': lc.bal_acc}).round(4))
display(Img(f'{ev}/confusion_matrices.png')); display(Img(f'{ev}/risk_coverage.png'))
pl = pd.read_csv(f'{ev}/per_label_recall.csv')
display(pl[pl.tier.str.startswith('T3')].sort_values('recall').head(15))
for g in sorted(glob.glob(f'{RUNS}/{BEST_RUN}/gradcam/*.png')):
    print(os.path.basename(g)); display(Img(g, width=420))
print('Tải về máy cho demo:', f'{DRIVE_DIR}/final/best_model.pth', 'và temperature.json')